# Walkthrough · the Aegean-scripts researcher: Linear A and Linear B

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryanpavlicek/pyaegean/blob/main/notebooks/walkthrough-aegean.ipynb)

*You work on the Aegean syllabic scripts.* Two of them can be read: Linear B and the Cypriot syllabary write Greek. Two cannot: Linear A and Cypro-Minoan are undeciphered, and no tool changes that. pyaegean is built around this asymmetry, and so is this notebook: readings where readings exist, labeled leads where they do not. It follows [Recipes → workflow F](https://github.com/ryanpavlicek/pyaegean/wiki/Recipes) (the Aegean-scripts researcher).

By the end you will have:

* the four bundled corpora loaded and their sign inventories inspected, including what is honestly unread;
* a 3,500-year-old ledger checked against its own stated total;
* exploratory structure (word families, conventional sound values) carrying the caveats that belong on it;
* Greek readings through the deciphered Linear B and Cypriot bridges;
* a citation for the exact corpus used.

**Offline-first.** All four corpora ship in the wheel; every ungated cell runs with no network. The one `RUN_HEAVY` cell fetches DAMOS, the full Mycenaean corpus (~3 MB once).

In [ ]:
# pyaegean works fully offline once installed. This installs it the first time
# you open the notebook in Colab; locally it's a no-op if it's already present.
try:
    import aegean
except ImportError:
    %pip install -q pyaegean
    import aegean

print("pyaegean", aegean.__version__)

for sid in ("lineara", "linearb", "cypriot", "cyprominoan"):
    c = aegean.load(sid)
    print(f"  {sid:12} {len(c.documents):5} bundled documents")
#   lineara       1721 bundled documents   (the full GORILA corpus)
#   linearb         18 bundled documents   (a sample; DAMOS below is the full corpus)
#   cypriot        180 bundled documents   (IG XV 1, with its apparatus)
#   cyprominoan      2 bundled documents   (a sample; signs only, undeciphered)

In [ ]:
# ── The one switch for the optional, downloading cells ──────────────────────
# Leave this False to keep the whole notebook offline. Flip it to True (on
# decent wifi) for the full Mycenaean corpus:
#   * aegean.load("damos")   ~3 MB one-time fetch — DAMOS, ~5,900 Linear B
#                            tablets (CC BY-NC-SA 4.0, fetched, never bundled)
# It fetches once, then caches. Every other cell runs offline with no downloads.
RUN_HEAVY = False

print("Heavy/optional cells are",
      "ON — first run downloads." if RUN_HEAVY
      else "OFF — the offline core still runs in full.")

## Step 1 · Sign inventories: read, and honestly unread

Every script plugin exposes its sign inventory: glyph, transliteration label, Unicode codepoint, and a sound value **where one is established**. The counts below are the honesty summary in one table. Linear B and Cypriot are deciphered. The Linear A values are conventional: they are carried over from the Linear B correspondence, which is the standard scholarly convention for transliteration, not a decipherment. Cypro-Minoan has no accepted values at all, and the inventory says so.

In [ ]:
from aegean import get_script

for sid in ("lineara", "linearb", "cypriot", "cyprominoan"):
    inv = get_script(sid).sign_inventory
    read = sum(1 for s in inv.signs if s.phonetic)
    print(f"  {sid:12} {len(inv.signs):3} signs · {read:3} with a sound value")
#   lineara      342 signs ·  50 with a sound value   (conventional, via Linear B)
#   linearb      211 signs ·  74 with a sound value   (deciphered)
#   cypriot       55 signs ·  55 with a sound value   (deciphered)
#   cyprominoan   99 signs ·   0 with a sound value   (undeciphered)

pa = get_script("linearb").sign_inventory.by_label("PA")
print("PA:", pa.glyph, hex(pa.codepoint), "·", pa.phonetic, "· Bennett", pa.attrs["bennett"])
# PA: 𐀞 0x1001e · pa · Bennett B003

s301 = get_script("lineara").sign_inventory.by_label("*301")
print("*301:", s301.glyph, "· phonetic:", s301.phonetic)
# *301: 𐙕 · phonetic: None
# A frequent Linear A sign with no established value: the inventory returns
# None rather than a guess, which is why *301 stays *301 in transliterations.

## Step 2 · A ledger that must add up (Haghia Triada, HT 13)

Linear A is undeciphered, but the *accounting shape* of many tablets is legible: ideograms, numerals (including fractions), and the recurring total word KU-RO. `parse_account_lines` reads a tablet as a ledger, and `balance_check` compares each stated total against the sum of the items above it.

The reconciliation is heuristic (section boundaries are inferred, and Linear A metrology is genuinely contested), so a discrepancy is a lead to inspect, never a verdict.

In [ ]:
from aegean.core import numerals

la = aegean.load("lineara")
doc = la.get("HT13")
print(doc.id, "·", doc.meta.site, "·", doc.meta.period)

lines = [[doc.tokens[i].text for i in line] for line in doc.lines]
for al in numerals.parse_account_lines(lines):
    print(f"  line {al.index}: {al.role:6} {' '.join(al.tokens):26} = {al.value}")
# HT13 · Haghia Triada · LMIB
#   line 0: header KA-U-DE-TA VIN 𐄁 TE 𐄁      = 0.0
#   line 1: item   RE-ZA 5 ¹⁄₂                = 5.5
#   line 2: item   TE-TU 56                   = 56.0
#   line 3: item   TE-KI 27 ¹⁄₂               = 27.5
#   line 4: item   KU-ZU-NI 18                = 18.0
#   line 5: item   DA-SI-*118 19              = 19.0
#   line 6: item   I-DU-NE-SI 5               = 5.0
#   line 7: total  KU-RO 130 ¹⁄₂              = 130.5

In [ ]:
from aegean.analysis import balance_check

for chk in balance_check(doc):
    print(chk)
# BalanceCheck(stated_total=130.5, computed_sum=131.0, item_count=6, difference=0.5,
#              balances=False, marker='KU-RO', total_line_index=7)
# The items sum to 131 but the scribe wrote 130½. Ancient error? A misread sign?
# An artefact of where we drew the section? A lead for a human.

checked = failing = 0
for d in la:
    for chk in balance_check(d):
        checked += 1
        failing += 0 if chk.balances else 1
print(f"{checked} checkable totals corpus-wide · {failing} do not reconcile")
# 41 checkable totals corpus-wide · 33 do not reconcile
# Most gaps trace to damage (item lines lost from the stone or clay), not to
# scribal arithmetic; each one is a pointer at a tablet worth reading.

## Step 3 · Exploratory structure, labeled as such

With no known language behind Linear A, distributional questions are still askable: which words share a stem and differ by a recurring ending? `find_morphological_clusters` answers exactly that and nothing more. A "suffix" here is a recurring final sign-string, not a confirmed morpheme, and a romanization like `jasasarame` is a conventional transliteration, not a pronunciation. pyaegean carries these caveats in the docstrings and in the output framing; the deciphered bridge functions simply do not exist for the undeciphered scripts, so the API cannot overclaim.

In [ ]:
from aegean.analysis import find_morphological_clusters
from aegean.scripts.lineara.phonetic import word_to_phonetic

clusters = find_morphological_clusters(la.word_frequencies())
print(len(clusters), "candidate word-families; the first:")
fam = clusters[0]
for m in fam.members[:4]:
    print(f"   {m.word:<16} ×{m.count}  +{m.suffix or '(stem)'}")
# 81 candidate word-families; the first:
#    JA-SA-SA-RA-ME   ×7  +SA-RA-ME     ← the libation-formula family
#    JA-SA            ×4  +(stem)
#    JA-SA-JA         ×1  +JA
#    JA-SA-MU         ×1  +MU

print("JA-SA-SA-RA-ME →", word_to_phonetic("JA-SA-SA-RA-ME"),
      "  (conventional sound values, not a reading)")
# JA-SA-SA-RA-ME → jasasarame

from aegean.scripts import lineara, linearb
print("greek_reading exists? lineara:", hasattr(lineara, "greek_reading"),
      "· linearb:", hasattr(linearb, "greek_reading"))
# greek_reading exists? lineara: False · linearb: True
# There is deliberately no Greek-reading bridge for an undeciphered script.

## Step 4 · The deciphered contrast: reading Linear B and Cypriot as Greek

For the two deciphered syllabaries the bridge is real: apply the known sound values plus the spelling conventions (unwritten final consonants, cluster handling) and the Greek word surfaces, with a gloss from the curated lexicon. Note `qa-si-re-u`: the q-series records a labiovelar consonant that Mycenaean still had and that later Greek turned into the β of βασιλεύς.

In [ ]:
from aegean.scripts import cypriot, linearb

for w in ("ti-ri-po", "ko-wo", "qa-si-re-u"):
    print(f"  {w:14} /{linearb.word_to_phonetic(w)}/ → {linearb.greek_reading(w)}")
#   ti-ri-po       /tiripo/ → ('τρίπους', 'tripod')
#   ko-wo          /kowo/ → ('κόρος', 'boy, son')
#   qa-si-re-u     /kwasireu/ → ('βασιλεύς', 'chief, local leader (later: king)')

w = "pa-si-le-u-se"
print(f"  {w:14} /{cypriot.word_to_phonetic(w)}/ → {cypriot.greek_reading(w)}")
#   pa-si-le-u-se  /pasileuse/ → ('βασιλεύς', 'king')
# The same word, five centuries later, in the other deciphered syllabary.

# For the undeciphered scripts the exploratory analogue is sound-matching.
# Calibrate it on a case where the answer is known:
from aegean.analysis import nearest

print(nearest("qa-si-re-u", "linearb",
              ["ποιμήν", "βασιλεύς", "πατήρ", "θεός", "δοῦλος"], "greek",
              top=2, fold_aspiration=True))
# [('βασιλεύς', 0.4375), ('πατήρ', 0.6625)]
# The true cognate ranks first by a clear margin. The RANKING is the signal:
# defective syllabic spelling inflates every absolute distance.

## Step 5 · Cypriot with its editorial apparatus (bundled, offline)

The bundled Cypriot corpus is the IG XV 1 syllabic material, and the edition's Leiden apparatus survives into Python: every token carries a `ReadingStatus`, so a restored royal title can be weighed differently from one read on the stone.

In [ ]:
from collections import Counter

cyp = aegean.load("cypriot")
d = cyp.get("IG XV 1, 120")
print(d.id, "·", d.meta.site, "·", d.meta.period)
for t in d.tokens:
    print(f"  {t.text:18} {t.status.name}")
print("e-mi →", cypriot.greek_reading("e-mi"))
# IG XV 1, 120 · Kurion · um 710-675 (I) / nach 498 (II)
#   a-ke-se-to-ro      CERTAIN      ← read on the stone
#   to                 CERTAIN
#   pa-po              CERTAIN
#   pa-si-le-wo-se     RESTORED     ← 'of the king': supplied by the editor
#   ti-mo-ke-re-to-se  RESTORED
#   e-mi               RESTORED
# e-mi → ('εἰμί', 'I am')

print("corpus-wide:", dict(Counter(t.status.name for dd in cyp.documents
                                   for t in dd.tokens).most_common()))
# corpus-wide: {'CERTAIN': 370, 'UNCLEAR': 188, 'RESTORED': 51, 'LOST': 19}

## Step 6 · Scale: the full Mycenaean corpus (optional fetch)

The bundled Linear B corpus is an 18-tablet sample. DAMOS (the Database of Mycenaean at Oslo, ~5,900 tablets, CC BY-NC-SA 4.0) fetches once and loads the same way, and the same balance check runs across all of it. On a corpus this fragmentary most checkable totals no longer reconcile, overwhelmingly because item lines are broken away, so read the count below as a damage survey and a worklist, not an audit of Mycenaean bookkeeping.

In [ ]:
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True to fetch the DAMOS corpus (~3 MB).")
else:
    damos = aegean.load("damos")          # ~5,900 tablets, fetched once, then cached
    checked = failing = 0
    for d in damos:
        for chk in balance_check(d):      # Linear B totals: TO-SO / TO-SA / TO-SO-DE
            checked += 1
            failing += 0 if chk.balances else 1
    print(len(damos.documents), "tablets ·", checked, "checkable totals ·",
          failing, "do not reconcile")
    print(damos.cite())
    # 5932 tablets · 255 checkable totals · 203 do not reconcile
    # Aurora, F. (2015). DAMOS (Database of Mycenaean at Oslo). Annotating a
    #   fragmentarily attested language. Procedia - Social and Behavioral
    #   Sciences, 198, 21-31. — https://damos.hf.uio.no

In [ ]:
print("pyaegean", aegean.__version__)
print("fingerprint:", la.fingerprint()[:16])
print(la.cite())
# fingerprint: c6e3d680a3a85842
# Godart, L. & Olivier, J.-P. (1976–1985). Recueil des inscriptions en linéaire A.
#   — https://github.com/mwenge/lineara.xyz

## Where to go next

* [Linear A](https://github.com/ryanpavlicek/pyaegean/wiki/Linear-A) and [Linear B](https://github.com/ryanpavlicek/pyaegean/wiki/Linear-B): each script's corpus, tools, and per-script limitations.
* [Cypriot](https://github.com/ryanpavlicek/pyaegean/wiki/Cypriot) and [Cypro-Minoan](https://github.com/ryanpavlicek/pyaegean/wiki/Cypro-Minoan): the other two syllabaries.
* [Recipes](https://github.com/ryanpavlicek/pyaegean/wiki/Recipes): workflow F chains these moves from the shell (`aegean balance lineara HT13`, `aegean analyze clusters lineara --top 3`, `aegean bridge linearb ko-wo`).
* [Analysis](https://github.com/ryanpavlicek/pyaegean/wiki/Analysis): the statistics behind the leads, including the null models that keep pattern-hunting honest.
* [Geography](https://github.com/ryanpavlicek/pyaegean/wiki/Geography): map any word's find-sites (`aegean geo lineara --word KU-RO`).
* [Limitations](https://github.com/ryanpavlicek/pyaegean/wiki/Limitations): the full, honest list.